# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [1]:

import nltk
import re
import pandas as pd
from pathlib import Path

from nltk.corpus import gutenberg

nltk.download("gutenberg", quiet=True)

# Misma normalización de la práctica anterior
def procesar_texto(texto):
    return re.findall(r"\b\w+\b", texto.lower())

# Conservar originales y textos procesados
documentos_originales = {} # para conservar los fragmentos en el ranking final
documentos = {} # almacenar los tokens para tf-idf

for doc_id in gutenberg.fileids():
    texto = gutenberg.raw(doc_id)

    documentos_originales[doc_id] = texto
    documentos[doc_id] = procesar_texto(texto)

print("Documentos cargados:", len(documentos))


Documentos cargados: 18


In [2]:

estadisticas = []

for doc_id, tokens in documentos.items():

    # Título abreviado derivado del nombre del archivo
    titulo = (
        Path(doc_id).stem
        .split("-", 1)[-1]
        .replace("_", " ")
        .title()
    )

    # Contar términos distintos
    unicos = len(set(tokens))

    # Longitud del documento
    longitud = len(tokens)

    estadisticas.append({
        "ID": doc_id,
        "Título abreviado": titulo,
        "Términos únicos": unicos,
        "Longitud": longitud
    })

df_documentos = pd.DataFrame(estadisticas)

display(df_documentos)


,ID,Título abreviado,Términos únicos,Longitud
0,austen-emma.txt,Emma,7256,161983
1,austen-persuasion.txt,Persuasion,5777,84167
2,austen-sense.txt,Sense,6336,120787
3,bible-kjv.txt,Kjv,12744,854046
4,blake-poems.txt,Poems,1512,6936
5,bryant-stories.txt,Stories,3900,46703
6,burgess-busterbrown.txt,Busterbrown,1539,16363
7,carroll-alice.txt,Alice,2571,27336
8,chesterton-ball.txt,Ball,8267,82867
9,chesterton-brown.txt,Brown,7739,73288


In [3]:

# Número de documentos
N = len(documentos)

# Construir el vocabulario global
# Debe contener todos los términos distintos
vocabulario = set()
for tokens in documentos.values():
    vocabulario.update(tokens)


# Longitudes documentales
longitudes = df_documentos["Longitud"]

print("Número de documentos:", N)
print("Tamaño del vocabulario:", len(vocabulario))

print("Longitud mínima:", longitudes.min())

# calcular la longitud media
print("Longitud media:", longitudes.mean())

print("Longitud máxima:", longitudes.max())


Número de documentos: 18
Tamaño del vocabulario: 42099
Longitud mínima: 6936
Longitud media: 122178.83333333333
Longitud máxima: 854046


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [7]:

from collections import Counter
from math import log

# 1. TF: contar apariciones por documento
def calcular_tf(tokens):
    return Counter(tokens)


# 2. DF: contar documentos que contienen cada término
def calcular_df(tf_por_documento):
    df = Counter()

    for frecuencias in tf_por_documento.values():
        # Actualizar DF usando los términos
        # únicos del documento (claves de frecuencias)
        df.update(frecuencias.keys())
    return df


# 3. IDF: calcular la rareza de cada término
def calcular_idf(df_por_termino, N):
    idf = {}

    for termino, frecuencia in df_por_termino.items():
        # Aplicar ln(N / DF) + 1
        idf[termino] = log((N ) / (frecuencia)) + 1 

    return idf


# 4. TF-IDF: peso de un término en un documento
def calcular_tfidf(termino, doc_id, tf_por_documento, idf):
    # Multiplicar TF por IDF
    return tf_por_documento[doc_id][termino] * idf[termino] 


In [8]:
# TF de todos los documentos
tf_por_documento = {
    doc_id: calcular_tf(tokens)
    for doc_id, tokens in documentos.items()
}

# DF global
df_por_termino = calcular_df(tf_por_documento)

# IDF global
idf_por_termino = calcular_idf(
    df_por_termino,
    len(documentos)
)

print("Documentos:", len(tf_por_documento))
print("Términos con DF:", len(df_por_termino))
print("Términos con IDF:", len(idf_por_termino))

Documentos: 18
Términos con DF: 42099
Términos con IDF: 42099


In [9]:
terminos = ["the", "emma", "whale", "love"]
doc_ejemplo = "austen-emma.txt"

tabla = []

for termino in terminos:
    tabla.append({
        "Término": termino,
        "TF": tf_por_documento[doc_ejemplo][termino],
        "DF": df_por_termino[termino],
        "IDF": idf_por_termino[termino],
        "TF-IDF": calcular_tfidf(
            termino, doc_ejemplo,
            tf_por_documento, idf_por_termino
        )
    })

df_verificacion = pd.DataFrame(tabla)

display(df_verificacion.round(4))

,Término,TF,DF,IDF,TF-IDF
0,the,5201,18,1.0000,5201.0000
1,emma,865,2,3.1972,2765.5993
2,whale,0,6,2.0986,0.0000
3,love,117,15,1.1823,138.3316


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [10]:

from scipy.sparse import csr_matrix

# Documentos y vocabulario en orden definido
ids = list(documentos.keys())
vocabulario_ordenado = sorted(vocabulario)

# Asociar cada término con su posición de columna
posicion_termino = {termino: i for i, termino in enumerate(vocabulario_ordenado)}

filas = []
columnas = []
pesos = []

# Recorrer documentos
for i, doc_id in enumerate(ids):

    # Recorrer términos y frecuencias de cada documento
    for termino, frecuencia in tf_por_documento[doc_id].items():

        filas.append(i)
        columnas.append(posicion_termino[termino])

        # Calcular el peso TF-IDF
        peso = calcular_tfidf(termino, doc_id, tf_por_documento, idf_por_termino)

        pesos.append(peso)

# Construir matriz dispersa
matriz_tfidf_manual = csr_matrix(
    (pesos, (filas, columnas)),
    shape=(len(ids), len(vocabulario_ordenado)),
    dtype=float
)

print("Dimensiones:", matriz_tfidf_manual.shape)
print("Valores no nulos:", matriz_tfidf_manual.nnz)


Dimensiones: (18, 42099)
Valores no nulos: 121988


In [11]:
doc_id = "austen-emma.txt"
termino = "emma"

fila = ids.index(doc_id)
columna = posicion_termino[termino]

peso_matriz = matriz_tfidf_manual[fila, columna]

print(f"Documento: {doc_id}")
print(f"Término: {termino}")
print(f"TF-IDF: {peso_matriz:.4f}")

Documento: austen-emma.txt
Término: emma
TF-IDF: 2765.5993


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [12]:

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Convertir los tokens en cadenas de texto
# respetando el orden de nuestros documentos
corpus_vectorizador = [
    " ".join(documentos[doc_id])
    for doc_id in ids
]

# Configurar el vectorizador
vectorizador = TfidfVectorizer(
    vocabulary=posicion_termino,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
    lowercase=False,
    token_pattern=r"(?u)\b\w+\b"
)

# Ajustar el vectorizador y transformar el corpus
matriz_tfidf_sklearn = vectorizador.fit_transform(corpus_vectorizador)

print("Matriz manual:", matriz_tfidf_manual.shape)
print("Matriz sklearn:", matriz_tfidf_sklearn.shape)


Matriz manual: (18, 42099)
Matriz sklearn: (18, 42099)


fit() aprende el vocabulario y calcula los IDF del corpus.

transform() convierte textos en vectores usando esos valores.

fit_transform() realiza ambas operaciones.

In [13]:

# Verificar las dimensiones
assert matriz_tfidf_manual.shape == matriz_tfidf_sklearn.shape

# TODO 2: Restar las matrices
diferencia = matriz_tfidf_manual - matriz_tfidf_sklearn

# Obtener la máxima diferencia absoluta
max_diferencia = (
    np.max(np.abs(diferencia.data))
    if diferencia.nnz > 0
    else 0.0
)

# Tolerancia numérica
tolerancia = 1e-8

print(f"Máxima diferencia absoluta: {max_diferencia:.12f}")
print("¿Matrices equivalentes?:", max_diferencia < tolerancia)


Máxima diferencia absoluta: 0.000000000000
¿Matrices equivalentes?: True


In [14]:
# PROBAR PESOS

fila = ids.index("austen-emma.txt")
columna = posicion_termino["emma"]

print(
    "TF-IDF manual:",
    matriz_tfidf_manual[fila, columna]
)

print(
    "TF-IDF sklearn:",
    matriz_tfidf_sklearn[fila, columna]
)

TF-IDF manual: 2765.59925939583
TF-IDF sklearn: 2765.59925939583


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [15]:

from collections import Counter
from scipy.sparse import csr_matrix, vstack

consultas = [
    {
        "consulta": "white whale captain",
        "necesidad": "Historias sobre ballenas y capitanes"
    },
    {
        "consulta": "alice rabbit wonderland",
        "necesidad": "Relatos sobre Alicia y sus aventuras"
    },
    {
        "consulta": "king murder revenge",
        "necesidad": "Obras sobre reyes, asesinatos y venganza"
    },
    {
        "consulta": "love marriage family",
        "necesidad": "Historias sobre amor y relaciones familiares"
    }
]

def vector_consulta_manual(consulta):
    frecuencias = Counter(procesar_texto(consulta))

    columnas = []
    pesos = []

    for termino, tf in frecuencias.items():
        if termino in posicion_termino:
            columnas.append(posicion_termino[termino])
            pesos.append(tf * idf_por_termino[termino])

    return csr_matrix(
        (pesos, ([0] * len(columnas), columnas)),
        shape=(1, len(vocabulario_ordenado))
    )

# Representación manual
vectores_consulta = vstack([
    vector_consulta_manual(c["consulta"])
    for c in consultas
])

# Representación mediante Scikit-learn
textos_consulta = [
    " ".join(procesar_texto(c["consulta"]))
    for c in consultas
]

vectores_sklearn = vectorizador.transform(textos_consulta)

print("Consultas:", len(consultas))
print("Matriz manual:", vectores_consulta.shape)
print("Matriz sklearn:", vectores_sklearn.shape)


Consultas: 4
Matriz manual: (4, 42099)
Matriz sklearn: (4, 42099)


In [16]:
# Calculo Manual
import numpy as np

# Producto punto: consultas × documentos
numeradores = (
    vectores_consulta @ matriz_tfidf_manual.T
).toarray()

# Normas de consultas y documentos
normas_q = np.sqrt(
    np.asarray(
        vectores_consulta.multiply(vectores_consulta).sum(axis=1)
    ).ravel()
)

normas_d = np.sqrt(
    np.asarray(
        matriz_tfidf_manual.multiply(matriz_tfidf_manual).sum(axis=1)
    ).ravel()
)

# calcular el denominador para cada pareja
denominadores = np.outer(normas_q, normas_d)

# division
# Evitar división entre cero con np.divide()
similitudes_manuales = np.divide(
    numeradores,
    denominadores,
    out=np.zeros_like(numeradores),
    where=denominadores != 0
)

print("Dimensiones de similitudes:", similitudes_manuales.shape)


Dimensiones de similitudes: (4, 18)


In [17]:
from sklearn.metrics.pairwise import cosine_similarity

similitudes_sklearn = cosine_similarity(
    vectores_sklearn,
    matriz_tfidf_sklearn
)

diferencia_maxima = np.max(
    np.abs(similitudes_manuales - similitudes_sklearn)
)

print(f"Diferencia máxima: {diferencia_maxima:.12f}")
print("¿Coinciden?:", np.allclose(
    similitudes_manuales,
    similitudes_sklearn,
    atol=1e-8
))

Diferencia máxima: 0.000000000000
¿Coinciden?: True


In [20]:

import pandas as pd

q = vectores_consulta.getrow(0)

doc_id = "melville-moby_dick.txt"
fila_doc = ids.index(doc_id)
d = matriz_tfidf_manual.getrow(fila_doc)

# Multiplicar componente a componente
productos = q.multiply(d).tocoo()

detalles = []

for columna, producto in zip(productos.col, productos.data):
    if producto != 0:
        detalles.append({
            "Término compartido": vocabulario_ordenado[columna],
            "Peso consulta": q[0, columna],
            "Peso documento": d[0, columna],
            "Producto parcial": producto
        })

tabla_detalles = pd.DataFrame(detalles)
display(tabla_detalles.round(4))

producto_punto = q.multiply(d).sum()
norma_q = np.sqrt(q.multiply(q).sum())
norma_d = np.sqrt(d.multiply(d).sum())

similitud = (
    producto_punto / (norma_q * norma_d)
    if norma_q > 0 and norma_d > 0
    else 0.0
)

print(f"Producto punto: {producto_punto:.4f}")
print(f"Norma consulta: {norma_q:.4f}")
print(f"Norma documento: {norma_d:.4f}")
print(f"Similitud coseno: {similitud:.6f}")

,Término compartido,Peso consulta,Peso documento,Producto parcial
0,captain,1.6931,557.0454,943.1599
1,whale,2.0986,2572.8987,5399.5168
2,white,1.1178,314.0970,351.0923


Producto punto: 6693.7690
Norma consulta: 2.9190
Norma documento: 21541.2714
Similitud coseno: 0.106456


¿Qué significa producto parcial?

Cada término compartido aporta una cantidad al producto punto:

$$
q\cdot d=\sum_i q_i d_i
$$

Por ejemplo, si `whale` tiene peso 3 en la consulta y peso 20 en el documento, su contribución es $$3\times20=60$$

La suma de las contribuciones de todos los términos compartidos forma el numerador de la similitud coseno.

| Concepto | Idea |
|---|---|
| Vector de consulta | Representación TF-IDF con el mismo vocabulario documental. |
| Producto punto | Suma las coincidencias ponderadas entre consulta y documento. |
| Norma | Mide la longitud del vector completo, incluidos términos que no aparecen en la consulta. |
| Similitud coseno | Divide el producto punto entre las normas para comparar orientaciones. |
| `cosine_similarity` | Permite verificar nuestra implementación manual. |


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [21]:
def extraer_fragmento(texto, consulta, longitud=180):
    # Unificar saltos de línea y espacios
    texto = re.sub(r"\s+", " ", texto)

    # Obtener términos de la consulta
    terminos = procesar_texto(consulta)

    # Buscar la primera aparición de alguno
    patron = r"\b(?:" + "|".join(map(re.escape, terminos)) + r")\b"
    coincidencia = re.search(patron, texto, re.IGNORECASE)

    # Extraer texto alrededor de la coincidencia
    inicio = max(0, coincidencia.start() - 60) if coincidencia else 0
    fin = min(len(texto), inicio + longitud)

    fragmento = texto[inicio:fin].strip()

    if inicio > 0:
        fragmento = "..." + fragmento

    if fin < len(texto):
        fragmento += "..."

    return fragmento

En lugar de mostrar todo un libro, presenta un fragmento de aproximadamente 180 caracteres cercano a una palabra buscada.

La función busca una coincidencia léxica, no necesariamente el fragmento más relevante del documento.

In [22]:

import numpy as np
import pandas as pd

# Relacionar identificadores y títulos
titulos = dict(zip(
    df_documentos["ID"],
    df_documentos["Título abreviado"]
))

rankings = {}

for i, consulta_info in enumerate(consultas):

    texto_consulta = consulta_info["consulta"]

    # Obtener puntuaciones de los 18 documentos
    puntuaciones = similitudes_manuales[i]

    # Ordenar puntuaciones de mayor a menor
    # y seleccionar las primeras cinco posiciones, argsorrt ordena  de menor a mayor, por eso se invierte con [::-1]
    orden = np.argsort(puntuaciones)[::-1][:5]

    resultados = []

    for posicion, j in enumerate(orden, start=1):

        doc_id = ids[j]

        resultados.append({
            "Posición": posicion,
            "Identificador": doc_id,
            "Título": titulos[doc_id],

            # Obtener puntuación del documento j
            "Puntuación": puntuaciones[j],

            "Fragmento": extraer_fragmento(
                documentos_originales[doc_id],
                texto_consulta
            )
        })

    rankings[texto_consulta] = pd.DataFrame(resultados)

    print(f"\nConsulta: {texto_consulta}")
    print("Necesidad:", consulta_info["necesidad"])

    display(rankings[texto_consulta].round(6))



Consulta: white whale captain
Necesidad: Historias sobre ballenas y capitanes


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,melville-moby_dick.txt,Moby Dick,0.106456,"...in hand to school others, and to teach them..."
1,2,austen-persuasion.txt,Persuasion,0.037169,"...pause which followed, added-- ""He is a rear..."
2,3,bryant-stories.txt,Stories,0.008759,...obody at all looked at Little Jack Rollarou...
3,4,blake-poems.txt,Poems,0.007157,...r sports shall be seen On the echoing Green...
4,5,chesterton-brown.txt,Brown,0.005726,...ing and fenced by little lemon and orange t...



Consulta: alice rabbit wonderland
Necesidad: Relatos sobre Alicia y sus aventuras


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,carroll-alice.txt,Alice,0.236410,[Alice's Adventures in Wonderland by Lewis Car...
1,2,burgess-busterbrown.txt,Busterbrown,0.005558,.... He felt that he had the greatest news to ...
2,3,bryant-stories.txt,Stories,0.001796,"...here yet, unless somebody picked him out ag..."
3,4,chesterton-thursday.txt,Thursday,0.000766,...o one of the little courts at the side of F...
4,5,chesterton-ball.txt,Ball,0.000436,"...o the sea! Shoo! Shoo! Shoo!"" The little ma..."



Consulta: king murder revenge
Necesidad: Obras sobre reyes, asesinatos y venganza


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,shakespeare-hamlet.txt,Hamlet,0.032839,...Nay answer me: Stand & vnfold your selfe Ba...
1,2,shakespeare-macbeth.txt,Macbeth,0.016519,...d filthie ayre. Exeunt. Scena Secunda. Alar...
2,3,bryant-stories.txt,Stories,0.014085,...in the jungle there was a great big Lion; a...
3,4,bible-kjv.txt,Kjv,0.012406,[The King James Bible] The Old Testament of th...
4,5,carroll-alice.txt,Alice,0.011039,"...e, 'they're sure to kill it in a day or two..."



Consulta: love marriage family
Necesidad: Historias sobre amor y relaciones familiares


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,blake-poems.txt,Poems,0.026154,"...arth a little space, That we may learn to b..."
1,2,austen-persuasion.txt,Persuasion,0.013830,"...proved it by adding, for the information of..."
2,3,austen-sense.txt,Sense,0.013032,[Sense and Sensibility by Jane Austen 1811] CH...
3,4,whitman-leaves.txt,Leaves,0.012918,"...ery ship that sails, sail you! Bear forth t..."
4,5,austen-emma.txt,Emma,0.010604,"..., indulgent father; and had, in consequence..."


In [23]:
# Comparacion

for i, consulta_info in enumerate(consultas):

    manual = similitudes_manuales[i]
    sklearn = similitudes_sklearn[i]

    coinciden = np.allclose(
        manual, sklearn, atol=1e-8
    )

    # Comparar también los primeros cinco
    top_manual = np.argsort(-manual)[:5]
    top_sklearn = np.argsort(-sklearn)[:5]

    print(f"Consulta: {consulta_info['consulta']}")
    print("Puntuaciones equivalentes:", coinciden)
    print(
        "Documentos Top-5 equivalentes:",
        set(top_manual) == set(top_sklearn)
    )
    print()

Consulta: white whale captain
Puntuaciones equivalentes: True
Documentos Top-5 equivalentes: True

Consulta: alice rabbit wonderland
Puntuaciones equivalentes: True
Documentos Top-5 equivalentes: True

Consulta: king murder revenge
Puntuaciones equivalentes: True
Documentos Top-5 equivalentes: True

Consulta: love marriage family
Puntuaciones equivalentes: True
Documentos Top-5 equivalentes: True



Debemos analizar si los documentos recuperados tienen sentido según la necesidad del usuario.

Por ejemplo, para white whale captain, esperamos que Moby Dick aparezca entre los resultados importantes porque contiene términos directamente relacionados con la consulta.

Sin embargo, nuestro método es léxico, no comprende necesariamente el significado completo de la consulta.

Esto implica que un documento con muchas coincidencias puede recibir una puntuación alta sin satisfacer completamente la necesidad del usuario.

| Concepto | Definición |
|-----------|------------|
| Score | Puntuación estimada de similitud para un documento. |
| Ranking | Orden de los documentos según el score. |
| Top-k | Primeros \(k\) documentos del ranking. |
| Evidencia textual | Fragmento del documento que permite examinar por qué puede ser relevante. |

1. Interpretación de los cuatro rankings

| Consulta | Primer documento | Similitud | Interpretación |
|-----------|------------------|------------|----------------|
| white whale captain | Moby Dick | 0.106456 | Coherente con la temática marítima y las ballenas. |
| alice rabbit wonderland | Alice's Adventures in Wonderland | 0.236410 | Coincidencia muy clara entre la consulta y el contenido. |
| king murder revenge | Hamlet | 0.032839 | Coherente con la temática de asesinato y venganza. |
| love marriage family | Poems | 0.026154 | Resultado menos evidente; las coincidencias léxicas favorecen al libro de poemas. |


La cuarta consulta es especialmente interesante: Poems supera a Persuasion, Sense and Sensibility y Emma, aunque las novelas de Austen podrían ser muy pertinentes para una búsqueda sobre matrimonio y relaciones familiares.

Esto demuestra una limitación del modelo: TF-IDF mide coincidencias ponderadas de términos, no comprensión semántica completa.

Además, una puntuación de 0.026 no representa un 2,6 % de relevancia. La similitud coseno es una medida geométrica que sirve principalmente para ordenar documentos según el modelo.

### Interpretación de los rankings TF-IDF

Se evaluaron cuatro consultas sobre los 18 documentos del corpus Gutenberg. Las similitudes calculadas manualmente coincidieron con las obtenidas mediante Scikit-learn, tanto en las puntuaciones como en los documentos del Top-5.

Los resultados principales fueron coherentes con las necesidades de información: *Moby Dick* ocupó la primera posición para la consulta sobre ballenas, *Alice's Adventures in Wonderland* para la consulta sobre Alicia y *Hamlet* para la búsqueda sobre asesinato y venganza.

Sin embargo, en la consulta sobre amor, matrimonio y familia, *Poems* obtuvo una puntuación superior a las novelas de Jane Austen. Esto evidencia que la similitud TF-IDF depende de las frecuencias y pesos de los términos, y no necesariamente de la relevancia semántica del documento.

Los fragmentos recuperados proporcionan evidencia textual, pero no siempre explican completamente la puntuación, ya que esta se calcula considerando todo el documento.

## 7. Sobre BM25

BM25 (Best Matching 25) es un modelo clásico de ranking que utiliza frecuencia de términos, frecuencia documental y longitud de documentos.

A diferencia de nuestro TF-IDF básico, BM25 controla cuánto aumenta la puntuación cuando un término aparece repetidamente y ajusta el efecto de la longitud del documento.

Su fórmula habitual, para una consulta sin términos repetidos, es:

$$BM25(q,d)=\sum\_{t\in q}IDF\_{BM25}(t)\frac{TF(t,d)(k_1+1)}{TF(t,d)+k_1(1-b+b\frac{|d|}{avgdl})}$$

Donde:

| Símbolo       | Significado                              |
| ------------- | ---------------------------------------- |
| (TF(t,d)) | Frecuencia del término en el documento.  |
| (\|d\|)   | Longitud del documento en tokens.        |
| (avgdl)   | Longitud promedio de los documentos.     |
| (k_1)     | Controla la saturación de la frecuencia. |
| (b)       | Controla la normalización por longitud.  |

Utilizaremos $$(k_1=1.5)$$ y $$(b=0.75)$$, dos valores convencionales.


In [27]:

import numpy as np

N = len(documentos)

longitud_promedio = np.mean([
    len(tokens) for tokens in documentos.values()
])

def calcular_bm25(consulta, doc_id, k1=1.5, b=0.75):

    longitud_doc = len(documentos[doc_id])
    frecuencias = tf_por_documento[doc_id]

    puntuacion = 0.0

    for termino in set(procesar_texto(consulta)):

        tf = frecuencias.get(termino, 0)

        if tf == 0:
            continue

        df = df_por_termino[termino]

        # calcular IDF de BM25
        idf = np.log(1 + (N - df + 0.5) / (df + 0.5))

        # 2: calcular la frecuencia normalizada
        frecuencia_normalizada = (
            tf * (k1 + 1) / ( tf + k1 * (1 - b + b * longitud_doc /longitud_promedio))
        )

        puntuacion += idf * frecuencia_normalizada

    return puntuacion


In [28]:

rankings_bm25 = {}

for consulta_info in consultas:

    texto_consulta = consulta_info["consulta"]

    puntuaciones = {
        doc_id: calcular_bm25(texto_consulta, doc_id)
        for doc_id in ids
    }

    top5 = sorted(
        ids,
        key=lambda doc_id: (-puntuaciones[doc_id], doc_id)
    )[:5]

    tabla = pd.DataFrame([
        {
            "Posición": i,
            "Documento": doc_id,
            "Título": titulos[doc_id],
            "BM25": puntuaciones[doc_id]
        }
        for i, doc_id in enumerate(top5, start=1)
    ])

    rankings_bm25[texto_consulta] = tabla

    print("\nConsulta:", texto_consulta)
    display(tabla.round(6))



Consulta: white whale captain


,Posición,Documento,Título,BM25
0,1,melville-moby_dick.txt,Moby Dick,4.746484
1,2,bryant-stories.txt,Stories,4.135172
2,3,whitman-leaves.txt,Leaves,3.999491
3,4,bible-kjv.txt,Kjv,2.669488
4,5,shakespeare-hamlet.txt,Hamlet,2.324150



Consulta: alice rabbit wonderland


,Posición,Documento,Título,BM25
0,1,carroll-alice.txt,Alice,11.065777
1,2,chesterton-thursday.txt,Thursday,6.253976
2,3,burgess-busterbrown.txt,Busterbrown,2.559289
3,4,bryant-stories.txt,Stories,2.481799
4,5,edgeworth-parents.txt,Parents,2.142633



Consulta: king murder revenge


,Posición,Documento,Título,BM25
0,1,melville-moby_dick.txt,Moby Dick,3.438168
1,2,edgeworth-parents.txt,Parents,3.322284
2,3,whitman-leaves.txt,Leaves,2.982907
3,4,milton-paradise.txt,Paradise,2.415288
4,5,bible-kjv.txt,Kjv,2.171520



Consulta: love marriage family


,Posición,Documento,Título,BM25
0,1,austen-sense.txt,Sense,2.351239
1,2,austen-persuasion.txt,Persuasion,2.341710
2,3,austen-emma.txt,Emma,2.328937
3,4,whitman-leaves.txt,Leaves,2.037007
4,5,chesterton-thursday.txt,Thursday,2.030603


In [29]:
resumen = []

for i, info in enumerate(consultas):
    consulta = info["consulta"]

    # Top-5 obtenido mediante TF-IDF
    orden_tfidf = np.argsort(-similitudes_manuales[i])[:5]
    top_tfidf = [ids[j] for j in orden_tfidf]

    # Top-5 obtenido mediante BM25
    top_bm25 = rankings_bm25[consulta]["Documento"].tolist()

    # Documentos compartidos por ambos rankings
    coincidencias = len(set(top_tfidf) & set(top_bm25))

    resumen.append({
        "Consulta": consulta,
        "Top-1 TF-IDF": titulos[top_tfidf[0]],
        "Top-1 BM25": titulos[top_bm25[0]],
        "Coincidencias Top-5": coincidencias,
        "Mismo Top-1": top_tfidf[0] == top_bm25[0]
    })

df_comparacion = pd.DataFrame(resumen)

display(df_comparacion)

,Consulta,Top-1 TF-IDF,Top-1 BM25,Coincidencias Top-5,Mismo Top-1
0,white whale captain,Moby Dick,Moby Dick,2,True
1,alice rabbit wonderland,Alice,Alice,4,True
2,king murder revenge,Hamlet,Moby Dick,1,False
3,love marriage family,Poems,Sense,4,False


## Conclusiones

En esta práctica se implementó un sistema básico de recuperación de información utilizando los **18 documentos del corpus Gutenberg de NLTK**, con un vocabulario de **42.099 términos únicos**.

Primero, se calcularon manualmente las frecuencias TF, DF e IDF y se construyó una matriz TF-IDF dispersa de dimensiones **18 × 42.099**, con **121.988 valores no nulos**.

Se comparó esta representación con `TfidfVectorizer` de Scikit-learn, obteniendo una diferencia máxima de cero dentro de la precisión numérica mostrada. Posteriormente, se implementó la similitud coseno mediante producto punto y normas vectoriales. Los resultados también coincidieron con `cosine_similarity`.

Se realizaron cuatro consultas y se construyeron rankings Top-5. Los resultados fueron coherentes en búsquedas específicas como *white whale captain* y *alice rabbit wonderland*, cuyos primeros documentos fueron *Moby Dick* y *Alice's Adventures in Wonderland*, respectivamente.

Finalmente, se implementó BM25 para comparar su comportamiento con TF-IDF. Se observaron diferencias importantes:

- En las consultas sobre ballenas y Alicia, ambos modelos coincidieron en el primer resultado.
- Para *king murder revenge*, TF-IDF posicionó primero a *Hamlet*, mientras que BM25 favoreció *Moby Dick*.
- Para *love marriage family*, BM25 priorizó novelas de Jane Austen, mientras que TF-IDF favoreció *Poems*.

**Se concluye que TF-IDF y BM25 son modelos de ranking léxico que producen resultados diferentes debido a sus mecanismos de ponderación.** BM25 incorpora saturación de frecuencia y normalización por longitud, mientras que el modelo TF-IDF implementado utiliza similitud coseno.

Aunque los resultados permiten identificar diferencias cualitativas, sería necesario contar con juicios de relevancia y métricas como Precision@5 o nDCG para establecer objetivamente cuál modelo ofrece mejor rendimiento.